# Environmental pressure pyramids

Extend inverted pyramids to environmental intensities (CO2, energy, water, land).

## Setup
This notebook assumes an MRIO system with satellite accounts loaded via `pymrio` (e.g., EXIOBASE3 provides emissions, energy, land, water).

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
if (ROOT / 'src').exists():
    sys.path.append(str(ROOT / 'src'))
elif (ROOT.parent / 'src').exists():
    sys.path.append(str(ROOT.parent / 'src'))

import pandas as pd
import ecolecon_mrio as mp
from ecolecon_mrio import data

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 20)


In [ ]:
from pathlib import Path
import pymrio

# dataset options: "oecd" (default), "test" (tiny demo), "exiobase" (public), "wiod" (requires IO file)
DATASET = "oecd"
YEAR = 2018
REGION = "DEU"  # set to None to use full system; for OECD, use country codes (e.g., DEU, FRA, GBR, ITA)

if DATASET == "wiod":
    data_dir = Path(f"data/wiod13_{YEAR}")
    pkl_path = data_dir / "mrio_system.pkl"
elif DATASET == "exiobase":
    data_dir = Path(f"data/exiobase3_{YEAR}")
    pkl_path = data_dir / "mrio_system.pkl"
elif DATASET == "oecd":
    data_dir = Path(f"data/oecd_icio_{YEAR}")
    pkl_path = data_dir / f"mrio_system_{YEAR}.pkl"
elif DATASET == "test":
    data_dir = Path("data/test")
    pkl_path = None
else:
    raise ValueError("Unsupported DATASET")

mrio = None
if pkl_path and pkl_path.exists():
    print(f"Loading cached MRIO from {pkl_path}")
    mrio = pymrio.read_mrio(pkl_path)
    try:
        mrio.calc_all()
    except Exception:
        pass
elif DATASET == "test":
    mrio = pymrio.load_test()
    mrio.calc_all()
else:
    data_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {DATASET.upper()} {YEAR} (may take minutes)...")
    if DATASET == "wiod":
        pymrio.download_wiod2013(storage_folder=data_dir, years=[YEAR])
        mrio = pymrio.parse_wiod(data_dir, year=YEAR)
    elif DATASET == "oecd":
        pymrio.download_oecd(storage_folder=data_dir, years=[YEAR])
        mrio = pymrio.parse_oecd(data_dir, year=YEAR)
    else:
        pymrio.download_exiobase3(storage_folder=data_dir, years=[YEAR], system="pxp")
        mrio = pymrio.parse_exiobase3(data_dir / f"IOT_{YEAR}_pxp.zip")
    mrio.calc_all()
    if pkl_path:
        mrio.save(pkl_path)
        print(f"Saved MRIO to {pkl_path}")


## Choose footprint
Select satellite account and build intensity (e.g., kg CO2 per monetary unit output).

In [ ]:
if mrio is not None:
    # Example for EXIOBASE3: CO2 satellite often named 'emissions' / 'co2'
    # Replace 'CO2' with the correct column in your satellite table
    satellite = mrio.satellite  # placeholder; set to the right DataFrame
    intensity = data.satellite_intensity(satellite, mrio.x)

    y = data.final_demand_vector(mrio)
    L = mp.compute_leontief(mrio.Z, mrio.x)
    pyramid_inputs = mp.PyramidInputs(leontief=L, final_demand=y, intensity=intensity)

In [ ]:
if mrio is not None:
    # Demand-driven (Leontief) already in data.
    L = mp.compute_leontief(mrio.Z, mrio.x)
    G = mp.compute_ghosh(mrio.Z, mrio.x)
    print("Leontief shape:", L.shape, "Ghosh shape:", G.shape)


## Embodied pressure base
Identify foundational sectors for the chosen footprint.

In [ ]:
if mrio is not None:
    pressure_base = mp.embodied_pressure(pyramid_inputs)
    print("Top sectors by embodied share:")
    display(mp.rank_foundational_sectors(pressure_base, top_n=20))

## Cross-footprint comparison
Repeat for multiple satellites (CO2, energy, water) and compare bases / overlaps.

In [ ]:
def build_pressure_base(mrio, satellite_df):
    intensity = data.satellite_intensity(satellite_df, mrio.x)
    y = data.final_demand_vector(mrio)
    L = mp.compute_leontief(mrio.Z, mrio.x)
    inputs = mp.PyramidInputs(leontief=L, final_demand=y, intensity=intensity)
    return mp.embodied_pressure(inputs)

pressure_results = {}
if mrio is not None:
    satellites = {
        "co2": mrio.satellite,  # replace with actual DataFrame handles
    }
    for name, sat_df in satellites.items():
        pressure_results[name] = build_pressure_base(mrio, sat_df)

pressure_results

## Visualisation hooks
Add plotting (bars/treemaps/network). Keep plotting functions in the library for reuse.

## Dependency graph and clustering
Supply-driven graph using Ghosh matrix with pruning, for environmental footprint lenses.

In [ ]:
if mrio is not None:
    ghosh_df = pd.DataFrame(mp.compute_ghosh(mrio.Z, mrio.x), index=mrio.Z.index, columns=mrio.Z.columns)
    dep_graph = mp.build_dependency_graph(
        ghosh_df,
        top_k_per_downstream=3,
        threshold=0.01,
        normalize_by_column=True,
    )
    undirected = mp.symmetrize(dep_graph, agg="max")
    communities = mp.detect_communities(undirected)
    print(f"Communities detected: {len(set(communities.values()))}")
